## House Prices (Data Preparation — Missing Values)

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from house_prices_mlops.pipeline import load_stage
from house_prices_mlops.data_tracker import DataTracker
from house_prices_mlops.preparation.missing import (
    normalize_structural_absence,
    normalize_masonry_absence,
)

df_before = load_stage("structure")
tracker = DataTracker.load(df_before)
display(tracker.profile)


2026-10-11 23:30:14.538 | INFO     | house_prices_mlops.config:<module>:11 - PROJ_ROOT path is: /home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops


,var,data_type,variable_type,unique_count,missing_count,missing_pct,zero_count,zero_pct,min,max,mean,median,std,skewness,kurtosis
0,Id,int64,identifier,1168,0,0.000000,0.0,0.0,1.0,1460.0,730.904966,732.5,425.369088,0.000920,-1.215850
1,MSSubClass,string,nominal,15,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MSZoning,object,nominal,5,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,LotFrontage,float64,continuous,107,217,18.578767,0.0,0.0,21.0,313.0,70.343849,70.0,24.897021,2.406787,19.454319
4,LotArea,int64,continuous,890,0,0.000000,0.0,0.0,1300.0,215245.0,10689.642123,9600.0,10759.366198,11.958088,186.444082
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,MoSold,int64,temporal,12,0,0.000000,0.0,0.0,1.0,12.0,6.356164,6.0,2.670707,0.241761,-0.371218
77,YrSold,int64,temporal,5,0,0.000000,0.0,0.0,2006.0,2010.0,2007.818493,2008.0,1.322639,0.082593,-1.187097
78,SaleType,object,nominal,9,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,SaleCondition,object,nominal,6,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Findings

In [ ]:
missing_findings = tracker.findings.loc[
    tracker.findings["stage"].eq("DU - Quality - Missing Values")
].copy()

display(missing_findings)


### Structural Categorical Absence

The original encodes structural categorical missing values as "None". The same categorical list and fillna("None") operation are retained inside normalize_structural_absence(). GarageYrBlt is preserved: absence has no construction year.

In [ ]:
df_structural = normalize_structural_absence(df_before)
columns = df_before.columns[
    df_before.isna().sum().ne(df_structural.isna().sum())
].tolist()
structural_review = pd.DataFrame({
    "before_missing": df_before[columns].isna().sum(),
    "after_missing": df_structural[columns].isna().sum(),
})
display(structural_review)


### MasVnrType and MasVnrArea

In [ ]:
masvnr_review = (
    df_before.loc[
        df_before["MasVnrType"].isna() | df_before["MasVnrArea"].isna(),
        ["MasVnrType", "MasVnrArea"],
    ].value_counts(dropna=False).reset_index(name="count")
)
display(masvnr_review)

df_individual = normalize_masonry_absence(df_structural)
mask_no_veneer = df_before["MasVnrType"].isna() & df_before["MasVnrArea"].eq(0)
print("Structural absence:", int(mask_no_veneer.sum()))
display(df_individual.loc[df_individual["MasVnrArea"].eq(0), "MasVnrType"].value_counts(dropna=False))


### Integrated Missing State

In [ ]:
df_hp = load_stage("missing")
pd.testing.assert_frame_equal(df_individual, df_hp)
assert len(df_hp) == len(df_before)
pd.testing.assert_series_equal(df_hp["Id"], df_before["Id"])
pd.testing.assert_frame_equal(df_before, load_stage("structure"))
changed_columns = [*columns, "MasVnrType"]
pd.testing.assert_frame_equal(
    df_hp.drop(columns=changed_columns),
    df_before.drop(columns=changed_columns),
)
tracker.update_profile(df_hp)


### Record Decisions

In [ ]:
for column in columns:
    before_missing = int(df_before[column].isna().sum())
    after_missing = int(df_hp[column].isna().sum())
    if before_missing != after_missing:
        tracker.log(
            var=column, stage="DP - Missing Values",
            action="encode_structural_absence", method="normalize_structural_absence",
            before=f"{before_missing} missing", after=f"{after_missing} missing",
        )

tracker.log(
    var="MasVnrType", stage="DP - Missing Values",
    action="encode_structural_absence", method="None_when_MasVnrArea_equals_0",
    before=f"{int(mask_no_veneer.sum())} missing with zero area",
    after=f"{int(mask_no_veneer.sum())} values encoded as None",
)
tracker.resolve_finding(
    key="mas_vnr_type_structural_absence",
    resolution="Los valores faltantes de MasVnrType con MasVnrArea igual a cero se representaron como ausencia estructural mediante None.",
)
tracker.resolve_finding(
    key="structural_missing_features",
    resolution="Los faltantes de las variables categóricas estructurales se representaron como None. Los faltantes de GarageYrBlt se conservaron porque la ausencia de garaje no tiene un año de construcción.",
)


### Model-Specific Imputation Candidates

The original evaluates LotFrontage median imputation by Neighborhood and Electrical mode imputation. These remain alternatives for experiment pipelines, fitted within each training fold. They are not applied to the common state or marked resolved here.

In [ ]:
lotfrontage_by_neighborhood = (
    df_before.groupby("Neighborhood")["LotFrontage"]
    .agg(["count", "median", "mean"]).sort_values("median")
)
display(lotfrontage_by_neighborhood)
display(df_before["Electrical"].value_counts(dropna=False))


### Ambiguous Missing Values

The original assumes structural absence when both MasVnrType and MasVnrArea are missing, later substitutes BrkFace for missing type with positive area, and fills GarageYrBlt with zero. These assumptions are not promoted to the common pipeline: unknown veneer values and missing construction years remain missing.

In [ ]:
display(df_hp.loc[df_hp["MasVnrArea"].isna(), ["Id", "MasVnrType", "MasVnrArea"]])
display(df_hp.loc[
    df_hp["MasVnrType"].isna() & df_hp["MasVnrArea"].gt(0),
    ["Id", "MasVnrType", "MasVnrArea"],
])
display(df_hp.loc[
    df_hp["GarageYrBlt"].isna(),
    ["GarageYrBlt", "GarageType", "GarageCars", "GarageArea"],
].value_counts(dropna=False))


### Remaining Missing Values

In [ ]:
plt.figure(figsize=(18, 8))
sns.heatmap(df_hp.isna(), cbar=False)
plt.title("After Common Missing Values Treatment")
plt.show()

remaining_missing = (
    df_hp.isna().sum().loc[lambda x: x > 0].sort_values(ascending=False)
)
display(remaining_missing)


### Save Findings

In [6]:
tracker.findings.loc[tracker.findings['stage'].eq('DU - Quality - Missing Values')]

,key,var,stage,finding,status,resolution
2,electrical_missing,Electrical,DU - Quality - Missing Values,1 valores faltantes (0.09%) en entrenamiento. ...,pending,None
11,lot_frontage_missing,LotFrontage,DU - Quality - Missing Values,18.58% de valores faltantes en entrenamiento. ...,pending,None
12,mas_vnr_area_missing,MasVnrArea,DU - Quality - Missing Values,6 registros de entrenamiento presentan tanto M...,pending,None
14,mas_vnr_type_structural_absence,MasVnrType,DU - Quality - Missing Values,672 registros de entrenamiento presentan MasVn...,pending,None
21,structural_missing_features,structural_missing_features,DU - Quality - Missing Values,Se identificaron valores faltantes estructural...,pending,None


In [ ]:
tracker.save()